In [9]:
# Importo librerias necesarias para analisis y modelado
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import *
import streamlit as st
import holidays
from calendar import monthrange
import os

In [10]:
# Cargo datos de inference para predicciones 2025
inferencia_df = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')

print(f"Inference shape: {inferencia_df.shape}")
inferencia_df.head()

Inference shape: (888, 13)


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [11]:
# Transformaciones para inference_df - mismas que en entrenamiento

# 1. Convertir fecha a datetime
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])

# 2. Filtrar solo noviembre (eliminar octubre) - hacer primero
inferencia_df = inferencia_df[inferencia_df['fecha'].dt.month == 11].copy()
print(f"Shape despues de filtrar noviembre: {inferencia_df.shape}")
print(f"Fechas: {inferencia_df['fecha'].min()} hasta {inferencia_df['fecha'].max()}")

# 3. Eliminar columnas que no necesitamos (targets y columnas que no iran al modelo)
inferencia_df = inferencia_df.drop(columns=['unidades_vendidas', 'ingresos'])
print(f"Shape despues de eliminar targets: {inferencia_df.shape}")

Shape despues de filtrar noviembre: (720, 13)
Fechas: 2025-11-01 00:00:00 hasta 2025-11-30 00:00:00
Shape despues de eliminar targets: (720, 11)


In [12]:
# 4. Variables temporales y de calendario (mismas que en entrenamiento)
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['dia'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.dayofweek
inferencia_df['semana_del_año'] = inferencia_df['fecha'].dt.isocalendar().week
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter
inferencia_df['dia_del_año'] = inferencia_df['fecha'].dt.dayofyear

# Nombre del dia
inferencia_df['nombre_dia'] = inferencia_df['fecha'].dt.day_name()

# Fin de semana (viernes 5, sabado 6, domingo 0)
inferencia_df['es_fin_semana'] = inferencia_df['dia_semana'].isin([5, 6]).astype(int)

# Festivos de España
es_holidays = holidays.Spain(years=inferencia_df['año'].unique())
inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(es_holidays).astype(int)

# Black Friday 2025 (4to jueves de noviembre)
def get_black_friday(year):
    nov = pd.date_range(start=f'{year}-11-01', periods=30, freq='D')
    thursday = [d for d in nov if d.weekday() == 3]
    return thursday[3] if len(thursday) > 3 else None

black_fridays = [get_black_friday(y) for y in inferencia_df['año'].unique() if get_black_friday(y)]
inferencia_df['es_black_friday'] = inferencia_df['fecha'].isin(black_fridays).astype(int)

# Cyber Monday
cyber_mondays = [bf + pd.Timedelta(days=3) for bf in black_fridays]
inferencia_df['es_cyber_monday'] = inferencia_df['fecha'].isin(cyber_mondays).astype(int)

# Comienzo y fin de mes
inferencia_df['es_comienzo_mes'] = (inferencia_df['dia_mes'] <= 5).astype(int)
inferencia_df['dia'] = inferencia_df['fecha'].dt.day
inferencia_df['es_fin_mes'] = (inferencia_df['dia_mes'] >= 25).astype(int)
inferencia_df['dia'] = inferencia_df['fecha'].dt.day

# Primer lunes del mes
def es_primer_lunes(row):
    primer_dia = pd.Timestamp(year=row['año'], month=row['mes'], day=1)
    primer_lunes = primer_dia + pd.Timedelta(days=(7 - primer_dia.weekday()) % 7)
    return 1 if row['fecha'] == primer_lunes else 0

inferencia_df['es_primer_lunes_mes'] = inferencia_df.apply(es_primer_lunes, axis=1)

# Rebajas de enero (no aplica para noviembre pero se incluye por consistencia)
inferencia_df['es_rebajas'] = ((inferencia_df['mes'] == 1) & (inferencia_df['dia_mes'] <= 31)).astype(int)
inferencia_df['dia'] = inferencia_df['fecha'].dt.day

print("Variables temporales y de calendario creadas")
print(f"Shape: {inferencia_df.shape}")

Variables temporales y de calendario creadas
Shape: (720, 28)


C:\Users\m26344676\AppData\Local\Temp\ipykernel_7460\1283389679.py:19: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(es_holidays).astype(int)


In [13]:
# 5. Cargar datos historicos para calcular lags
# Los lags se calculan con datos historicos de 2024 (mismo producto y mismo periodo del año anterior)

ventas_2024 = pd.read_csv('../data/raw/entrenamiento/ventas.csv')
ventas_2024['fecha'] = pd.to_datetime(ventas_2024['fecha'])

# Obtener datos de 2024 para los mismos productos
ventas_2024 = ventas_2024[ventas_2024['producto_id'].isin(inferencia_df['producto_id'].unique())]

# Crear key para merge (producto + mes + dia)
ventas_2024['key'] = ventas_2024['producto_id'] + '_' + ventas_2024['fecha'].dt.month.astype(str) + '_' + ventas_2024['fecha'].dt.day.astype(str)
inferencia_df['key'] = inferencia_df['producto_id'] + '_' + inferencia_df['mes'].astype(str) + '_' + inferencia_df['dia_mes'].astype(str)

# Merge para obtener unidades_vendidas del año anterior
ventas_2024_rename = ventas_2024[['key', 'unidades_vendidas']].rename(columns={'unidades_vendidas': 'lag_1'})
inferencia_df = inferencia_df.merge(ventas_2024_rename, on='key', how='left')

# Para los demas lags, usamos el mismo valor (aproximacion para inference)
# En un caso real, se necesitarian mas datos historicos
for lag in range(2, 8):
    inferencia_df[f'lag_{lag}'] = inferencia_df['lag_1']

# Rolling mean de 7 dias (usamos el mismo valor como aproximacion)
inferencia_df['rolling_mean_7'] = inferencia_df['lag_1']

# Eliminar columna key temporal
inferencia_df = inferencia_df.drop(columns=['key'])

# Llenar NaN con 0 si no hay datos historicos
columnas_lags = [f'lag_{i}' for i in range(1, 8)] + ['rolling_mean_7']
inferencia_df[columnas_lags] = inferencia_df[columnas_lags].fillna(0)

print(f"Lags creados")
print(f"Shape: {inferencia_df.shape}")
inferencia_df[columnas_lags].describe()

Lags creados
Shape: (2880, 36)


,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,rolling_mean_7
count,2880.000000,2880.000000,2880.000000,2880.000000,2880.000000,2880.000000,2880.000000,2880.000000
mean,5.257292,5.257292,5.257292,5.257292,5.257292,5.257292,5.257292,5.257292
std,6.807285,6.807285,6.807285,6.807285,6.807285,6.807285,6.807285,6.807285
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
50%,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
75%,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
max,85.000000,85.000000,85.000000,85.000000,85.000000,85.000000,85.000000,85.000000


In [14]:
# Agregar columnas de precio que necesita el modelo
# Calcular precio_competencia (promedio de los 3 competidores)
inferencia_df['precio_competencia'] = inferencia_df[['Amazon', 'Decathlon', 'Deporvillage']].mean(axis=1)

# Calcular descuento_pct (basado en precio_venta actual vs precio_base)
inferencia_df['descuento_pct'] = ((inferencia_df['precio_venta'] - inferencia_df['precio_base']) / inferencia_df['precio_base']) * 100

# Calcular ratio_precio
inferencia_df['ratio_precio'] = inferencia_df['precio_venta'] / inferencia_df['precio_competencia']

print('Columnas de precio creadas')
print(inferencia_df[['precio_base', 'precio_venta', 'precio_competencia', 'descuento_pct', 'ratio_precio']].head())

Columnas de precio creadas
   precio_base  precio_venta  precio_competencia  descuento_pct  ratio_precio
0          115         115.0              100.61            0.0      1.143028
1          115         115.0              100.61            0.0      1.143028
2          115         115.0              100.61            0.0      1.143028
3          115         115.0              100.61            0.0      1.143028
4          135         135.0              119.08            0.0      1.133692


In [15]:
# Renombrar columnas de lags para que coincidan con el modelo
rename_dict = {
    "lag_1": "unidades_vendidas_lag_1",
    "lag_2": "unidades_vendidas_lag_2",
    "lag_3": "unidades_vendidas_lag_3",
    "lag_4": "unidades_vendidas_lag_4",
    "lag_5": "unidades_vendidas_lag_5",
    "lag_6": "unidades_vendidas_lag_6",
    "lag_7": "unidades_vendidas_lag_7",
    "rolling_mean_7": "unidades_vendidas_ma7"
}

inferencia_df = inferencia_df.rename(columns=rename_dict)

print("Columnas renombradas para lags")

Columnas renombradas para lags


In [16]:
# 6. One-hot encoding (mismo que en entrenamiento)
# Crear copias para one-hot
inferencia_df['nombre_h'] = inferencia_df['nombre']
inferencia_df['categoria_h'] = inferencia_df['categoria']
inferencia_df['subcategoria_h'] = inferencia_df['subcategoria']

# Aplicar one-hot encoding
inferencia_df = pd.get_dummies(inferencia_df, columns=['nombre_h', 'categoria_h', 'subcategoria_h'], drop_first=False)

print(f"One-hot encoding aplicado")
print(f"Shape final: {inferencia_df.shape}")

One-hot encoding aplicado
Shape final: (2880, 83)


In [17]:
# 7. Alinear columnas con el modelo (eliminar columnas que no usa el modelo)
# Columnas a eliminar (mismas que en entrenamiento)
cols_to_drop = ['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria', 'nombre_dia']
inferencia_df = inferencia_df.drop(columns=cols_to_drop)

print(f"Columnas eliminadas: {cols_to_drop}")
print(f"Shape final: {inferencia_df.shape}")
print(f"\nColumnas finales:")
print(inferencia_df.columns.tolist())

Columnas eliminadas: ['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria', 'nombre_dia']
Shape final: (2880, 77)

Columnas finales:
['precio_base', 'es_estrella', 'precio_venta', 'Amazon', 'Decathlon', 'Deporvillage', 'año', 'mes', 'dia_mes', 'dia', 'dia_semana', 'semana_del_año', 'trimestre', 'dia_del_año', 'es_fin_semana', 'es_festivo', 'es_black_friday', 'es_cyber_monday', 'es_comienzo_mes', 'es_fin_mes', 'es_primer_lunes_mes', 'es_rebajas', 'unidades_vendidas_lag_1', 'unidades_vendidas_lag_2', 'unidades_vendidas_lag_3', 'unidades_vendidas_lag_4', 'unidades_vendidas_lag_5', 'unidades_vendidas_lag_6', 'unidades_vendidas_lag_7', 'unidades_vendidas_ma7', 'precio_competencia', 'descuento_pct', 'ratio_precio', 'nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23', 'nombre_h_Asics Gel Nimbus 25', 'nombre_h_Bowflex SelectTech 552', 'nombre_h_Columbia Silver Ridge', 'nombre_h_Decathlon Bandas Elásticas Set', 'nombre_h_Domyos BM900', 'nombre_h_Domyos Kit Mancuernas 

In [18]:
# 8. Guardar dataframe transformado
os.makedirs('../data/processed', exist_ok=True)
inferencia_df.to_csv('../data/processed/inferencia_df_transformado.csv', index=False)

print(f"Dataframe guardado en: ../data/processed/inferencia_df_transformado.csv")
print(f"Shape: {inferencia_df.shape}")
inferencia_df.head()

Dataframe guardado en: ../data/processed/inferencia_df_transformado.csv
Shape: (2880, 77)


,precio_base,es_estrella,precio_venta,Amazon,Decathlon,Deporvillage,año,mes,dia_mes,dia,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
0,115,True,115.0,81.78,117.08,102.97,2025,11,1,1,...,False,False,False,False,False,False,False,False,True,False
1,115,True,115.0,81.78,117.08,102.97,2025,11,1,1,...,False,False,False,False,False,False,False,False,True,False
2,115,True,115.0,81.78,117.08,102.97,2025,11,1,1,...,False,False,False,False,False,False,False,False,True,False
3,115,True,115.0,81.78,117.08,102.97,2025,11,1,1,...,False,False,False,False,False,False,False,False,True,False
4,135,True,135.0,121.00,116.13,120.11,2025,11,1,1,...,False,False,False,False,False,False,False,False,True,False
